# 07 — Open Data Mobility: exploration notebook

**Next-hour NYC taxi-zone demand forecasting** on real, messy public data —
41.2M TLC yellow-taxi trips (2024) joined with NOAA GHCN-Daily weather,
aggregated to a zero-filled 265 zone × 8,784 hour grid.

This notebook demonstrates the project's essence in a few minutes:

1. **Load the prepared artifacts** — curated zone×hour demand and persisted
   holdout predictions (no re-download, no retraining).
2. **The headline comparison** — the run ladder (naive vs GBM vs TemporalNet)
   straight from `reports/metrics.json`, plus baseline deltas.
3. **One key result, reproduced** — p90 uncertainty-band coverage recomputed
   from persisted predictions with the package's own evaluation helpers.
4. **Weather elasticity** — the rain-vs-dry demand ratio per borough, the
   directional (not causal) signal a planner would act on.

## How to run

From the repo root:

```bash
uv sync --all-packages
```

then run the cells of this notebook **from the project directory**
(`projects/07-open-data-mobility`). Every input is a persisted artifact
under `data/` and `reports/` — produced by `dvc repro` — so no cell
touches the network and none takes more than a few seconds. If the
artifacts are missing, run `uv run dvc repro` from the project dir first.

## 1. Load the prepared data

The pipeline never materialises raw trips; what we load here is the curated
output: the complete, zero-filled zone×hour grid, and the holdout
predictions every model wrote during training. Paths come from the
package's own validated config, so this works from any working directory.

In [ ]:
import polars as pl
from mobility.config import get_config

cfg = get_config()

demand = pl.read_parquet(cfg.zone_hour_demand_path)
preds = pl.read_parquet(cfg.models_dir / "predictions.parquet")

print(
    f"curated grid : {demand.height:,} zone-hours "
    f"({demand['zone_id'].n_unique()} zones x {demand['ts'].n_unique():,} hours)"
)
print(f"holdout preds: {preds.height:,} zone-hours, {preds['ts'].min()} .. {preds['ts'].max()}")
print(f"prediction columns: {[c for c in preds.columns if c.startswith('y_')]}")

## 2. Headline comparison — the run ladder

`reports/metrics.json` is the single source of truth the case study renders.
Here it is as a table, with the ΔMAE vs the `naive_lag24` persistence
baseline — the acceptance gate was ≥ 20% MAE reduction.

In [ ]:
import json

metrics = json.loads((cfg.reports_dir / "metrics.json").read_text())
meta = metrics["_meta"]

rows = []
for run, m in metrics.items():
    if run.startswith("_") or run == "deltas":
        continue
    delta = metrics["deltas"].get(run, {})
    rows.append(
        {
            "run": run,
            "MAE": round(m["mae"], 3),
            "RMSE": round(m["rmse"], 3),
            "MAPE %": round(m["mape_pct"], 1),
            "ΔMAE vs naive %": delta.get("mae_reduction_pct", "—"),
        }
    )

print(f"holdout: {meta['split']} ({meta['n_test_rows']:,} zone-hours)\n")
print(pl.DataFrame(rows))

## 3. Reproduce one key result — p90 band calibration

The deliverable that turns a forecast into a staffing number is the p90
upper band: it should contain ~90% of realised demand. The packaged
`p90_coverage*` helpers recompute it from the persisted predictions in
seconds — and land on the same 90.8% the report ships.

In [ ]:
from mobility.evaluate import p90_coverage, p90_coverage_by_borough

p90_col = "y_p90_tuned" if "y_p90_tuned" in preds.columns else "y_p90"

recomputed = p90_coverage(preds, p90_col)
shipped = meta["p90_coverage"]
print(f"p90 coverage recomputed from predictions: {recomputed:.4f}")
print(f"p90 coverage shipped in metrics.json  : {shipped:.4f}")
assert abs(recomputed - shipped) < 1e-3, "reproduction mismatch"

print("\nper borough (target 0.90):")
print(p90_coverage_by_borough(preds, p90_col))

## 4. Weather elasticity — what rain does to curb demand

Rain vs dry mean hourly demand per borough, from an honest groupby over the
holdout window. **Directional, not causal**: the hour-of-week mix is not
normalised, so treat these as elasticity signals, not pricing triggers.

In [ ]:
from mobility.evaluate import weather_elasticity

elasticity = weather_elasticity(preds, demand)
print(
    elasticity.select(
        [
            "borough",
            "mean_pickups_rain",
            "mean_pickups_dry",
            "rain_vs_dry_ratio",
            "n_hours_rain",
            "n_hours_dry",
        ]
    )
)

## Where to go next

- **Case study:** [/projects/07-open-data-mobility](/projects/07-open-data-mobility) —
  the full narrative: EDA, training protocol, evaluation setup, results.
- **Design:** [DESIGN.md](../DESIGN.md) — bounded slice, DQ contracts, split
  discipline, and the audit-trail rationale behind every choice.
- **Code:** [`src/mobility/`](../src/mobility/) — streaming Polars pipeline
  (`download` → `validate` → `join` → `features`), GBM/TemporalNet training
  with Optuna tuning, and the evaluation/reporting module used above.

Everything here was produced by `dvc repro` from public, license-clean data —
re-runnable end-to-end in minutes on a laptop.